# EMTscore Analysis and Plots (automated)

This notebook is a thin front-end. Every cell is a one-liner call into
`emtscore.workflow`, which contains the same logic as
`EMTScore_workflow_clean_v2.ipynb` (heavy code lifted out into Python_Conv).

## 2. Example Walkthrough (Bulk RNA-seq)


### 0. Environment

In [1]:
# Make the Python_Conv package importable from anywhere
import sys
from pathlib import Path
PKG_ROOT = Path.cwd().resolve()
while not (PKG_ROOT / 'emtscore').is_dir():
    if PKG_ROOT == PKG_ROOT.parent:
        break
    PKG_ROOT = PKG_ROOT.parent
sys.path.insert(0, str(PKG_ROOT))
print('Package root:', PKG_ROOT)


Package root: C:\Users\shawn\OneDrive\Github Repo\PythonLab\Python_Conv


In [ ]:
%matplotlib inline
from IPython.display import display
from emtscore import workflow as wf


### 2.1 - 2.4 Load cell annotation, gene expression, signatures, GMT


In [ ]:
inp = wf.load_inputs()
display(inp.cell_annot.head())
display(inp.geneExp.iloc[:4, :5])
display(inp.E_sig.head())
display(inp.M_sig.head())


### 2.5 Compute Scores Using Multiple Methods


In [ ]:
scores = wf.score_all_methods(inp.geneExp, inp.gmt_path)
display(scores.comparison.head())


### 2.6 Prepare Data for Plotting


In [ ]:
pd_data = wf.prepare_plot_dataframes(scores, inp.geneExp, inp.M_sig, inp.cell_annot)


### 2.7 E Score vs M Score Plot

Scatter plot with KDE contours and mean ± SD markers per cell type.


In [ ]:
print(pd_data.nnPCA_em.head())


In [ ]:
# Re-align indices and recompute nnPCA per gene set (v2 cell 16)
rb = wf.rebuild_em_for_plot(inp.geneExp, inp.cell_annot, inp.gmt_path)


In [ ]:
# Three E-vs-M panels: nnPCA, AUCell, ssGSEA
for fig in wf.plot_em_section(rb):
    if fig is not None:
        display(fig)


### 2.8 M1 vs M2 Score Plot


In [ ]:
display(wf.plot_m1_m2(pd_data.nnPCA_mm))


### 2.9 Combined Plot (E vs M  |  M1 vs M2)


In [ ]:
display(wf.plot_combined_em_m1_m2(rb.nnPCA_em, pd_data.nnPCA_mm))


### 2.10 E/M Score Distributions (M1 Histogram)


In [ ]:
display(wf.plot_m1_histogram(pd_data.nnPCA_mm))


### 2.11 Heatmap (M Signature Genes, Hierarchically Clustered)


#### 2.11.0 Full M-Signature Heatmap (All Genes)


In [ ]:
display(wf.plot_full_m_heatmap(rb.geneExp, inp.M_sig, rb.nnPCA_em))


In [ ]:
display(wf.plot_pc_driver_heatmap(rb.geneExp, inp.M_sig, rb.nnPCA_em, pd_data.nnPCA_mm))


---
## 3. Example Walkthrough: Single-Cell RNA-seq


### 3.1 Data Importing and Basic Calculations


In [ ]:
adatas = wf.load_cook_adatas()


#### 3.1.1 EMT score across pseudotime per condition


In [ ]:
display(wf.plot_emt_vs_pseudotime(['A549_EGF', 'A549_TGFB1']))


### 3.2 Building Gaussian Mixture Models (GMM) in E-M Space


In [ ]:
gmm_results = wf.build_gmm_in_em_space()


#### 3.2.1 Sankey: GMM cluster vs true timepoint label


In [ ]:
for fig in wf.plot_gmm_sankey(gmm_results, ['A549_TGFB1', 'A549_EGF']):
    display(fig)


### 3.3 Identifying Pathways Most Associated with the EMT Process


In [ ]:
result = wf.run_pathway_correlation_v2()
display(result.head(10))


#### 3.3.1 Top 10 Positively Correlated Pathways


In [ ]:
wf.plot_top_pathways(result, n=10, mode='positive',
                     title='Top 10 Positive Correlated Pathways').show()


#### 3.3.2 Top 10 Negatively Correlated Pathways


In [ ]:
wf.plot_top_pathways(result, n=10, mode='negative',
                     title='Top 10 Negative Correlated Pathways').show()


### 3.4 E vs M and PC1 vs PC2 Scatter Plots


In [ ]:
fig_a, fig_b, fig_c, adata_sc = wf.plot_em_pc_panels_cook()
display(fig_a); display(fig_b); display(fig_c)


### 3.5 Computing Additional Scores (Stemness and Senescence)


In [ ]:
adata_sc = wf.compute_stem_senescence(adata_sc)


### 3.6 Relationship Between Stemness, Senescence, and EMT Scores


In [ ]:
display(wf.plot_stemness_vs_senescence(adata_sc))


In [ ]:
display(wf.plot_em_vs_stem_sen(adata_sc))
